
<div  style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://raw.githubusercontent.com/derar-alhussein/Databricks-Certified-Data-Engineer-Associate/main/Includes/images/bookstore_schema.png" alt="Databricks Learning" style="width: 600">
</div>

In [0]:
%run ../Includes/Copy-Datasets


## Exploring The Source Directory

In [0]:
files = dbutils.fs.ls(f"{dataset_bookstore}/orders-raw")
display(files)


## Auto Loader

In [0]:
(spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", "dbfs:/mnt/demo/orders_checkpoint")
        .load(f"{dataset_bookstore}/orders-raw")
      .writeStream
        .option("checkpointLocation", "dbfs:/mnt/demo/orders_checkpoint")
        .table("orders_updates")
)

In [0]:
%sql
SELECT * FROM orders_updates

In [0]:
%sql
SELECT count(*) FROM orders_updates


## Landing New Files

In [0]:
load_new_data()

In [0]:
files = dbutils.fs.ls(f"{dataset_bookstore}/orders-raw")
display(files)

In [0]:
%sql
SELECT count(*) FROM orders_updates


## Exploring Table History

In [0]:
%sql
DESCRIBE HISTORY orders_updates


## Cleaning Up

In [0]:
%sql
DROP TABLE orders_updates

In [0]:
dbutils.fs.rm("dbfs:/mnt/demo/orders_checkpoint", True)

# Auto Loader Configurations 


## cloudFiles.useNotifications

In [0]:
# cloudFiles.useNotifications to true switches Auto Loader to file notification mode
(spark.readStream.format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", f"{cloud_storage_path}/orders_checkpoint")
  .option("cloudFiles.useNotifications", "true")
  .load(f"{dataset_bookstore}/orders-raw")
  .writeStream
  .option("checkpointLocation", f"{cloud_storage_path}/orders_checkpoint")
  .table("orders")

## cloudFiles.allowOverwrites

In [0]:
# Setting cloudFiles.allowOverwrites to true instructs Auto Loader to look at both the file path and the file's last-modified timestamp. When a file is overwritten or re-uploaded, Auto Loader detects the updated timestamp and re-ingests the entire file into the stream
spark.readStream.format("cloudFiles")
     .option("cloudFiles.format", "json")
     .option("cloudFiles.allowOverwrites", "true")
     .load(cloud_storage_path)


## Files filters: pathGlobFilter 

In [0]:
# To filter input files based on a specific pattern, such as *.png, you can use the pathGlobFilter option
spark.readStream.format("cloudFiles")
     .option("cloudFiles.format", "binaryFile")
     .option("pathGlobfilter", "*.png")
     .load("/path/to/files"

## Schema inference:cloudFiles.inferColumnTypes

In [0]:
# Setting cloudFiles.inferColumnTypes to true changes this default behavior. It instructs Auto Loader to sample the initial batch of files and select precise, specific data types for columns (such as integers, booleans, and timestamps) 
spark.readStream.format("cloudFiles")
     .option("cloudFiles.format", "json")
     .option("cloudFiles.schemaLocation", path)
     .option("cloudFiles.inferColumnTypes", "true")
     .load(input_path)

## Schema evolution: cloudFiles.schemaEvolutionMode

In [0]:
#Auto Loader detects the addition of new columns in input files during processing. To control how this schema change is handled,
spark.readStream
     .format("cloudFiles")
     .option("cloudFiles.format", <source_format>)
     .option("cloudFiles.schemaLocation", "<schema_path>")
     .option("cloudFiles.schemaEvolutionMode", <mode>)
     .load("/path/to/files")

![image_1787196827122.png](./image_1787196827122.png "image_1787196827122.png")

The default mode is **addNewColumns**, so when Auto Loader detects a new column, the stream stops with an UnknownFieldException. Before your stream throws this error, Auto Loader updates the schema location with the latest schema by merging new columns to the end of the schema. The next run of the stream executes successfully with the updated schema.